In [12]:
import pandas as pd
from itertools import combinations
from statsmodels.stats.multitest import multipletests
from scipy import stats

In [13]:
experiment_m = pd.read_csv("../data/ab_testing/experiment_m.csv")
groups = sorted(experiment_m["group"].dropna().unique())

groups

['Test1',
 'Test2',
 'Test3',
 'Test4',
 'Test5',
 'Test6',
 'Test7',
 'Test8',
 'Test9',
 'control']

In [14]:
for i in combinations(groups, 2) :
    print(i)

('Test1', 'Test2')
('Test1', 'Test3')
('Test1', 'Test4')
('Test1', 'Test5')
('Test1', 'Test6')
('Test1', 'Test7')
('Test1', 'Test8')
('Test1', 'Test9')
('Test1', 'control')
('Test2', 'Test3')
('Test2', 'Test4')
('Test2', 'Test5')
('Test2', 'Test6')
('Test2', 'Test7')
('Test2', 'Test8')
('Test2', 'Test9')
('Test2', 'control')
('Test3', 'Test4')
('Test3', 'Test5')
('Test3', 'Test6')
('Test3', 'Test7')
('Test3', 'Test8')
('Test3', 'Test9')
('Test3', 'control')
('Test4', 'Test5')
('Test4', 'Test6')
('Test4', 'Test7')
('Test4', 'Test8')
('Test4', 'Test9')
('Test4', 'control')
('Test5', 'Test6')
('Test5', 'Test7')
('Test5', 'Test8')
('Test5', 'Test9')
('Test5', 'control')
('Test6', 'Test7')
('Test6', 'Test8')
('Test6', 'Test9')
('Test6', 'control')
('Test7', 'Test8')
('Test7', 'Test9')
('Test7', 'control')
('Test8', 'Test9')
('Test8', 'control')
('Test9', 'control')


In [15]:
group_counts = (
    experiment_m.groupby("group", as_index=False)
    .size()
    .rename(columns={"size": "n"})
)

group_counts

,group,n
0,Test1,200
1,Test2,200
2,Test3,200
3,Test4,200
4,Test5,200
5,Test6,200
6,Test7,200
7,Test8,200
8,Test9,200
9,control,200


In [16]:
experiment_m.head()

,viewing_time,group
0,15.656666,control
1,70.875219,control
2,31.530482,control
3,66.033616,control
4,34.834495,control


In [24]:
def pairwise_ttests(data, value_col, group_col):
    groups = sorted(data[group_col].dropna().unique())
    rows = []

    for g1, g2 in combinations(groups, 2):
        x = data.loc[data[group_col] == g1, value_col].dropna()
        y = data.loc[data[group_col] == g2, value_col].dropna()

        t_stat, raw_p = stats.ttest_ind(
            x, y,
            equal_var=False,
            nan_policy="omit"
        )
    
        rows.append({
        "group_1": g1,
        "group_2": g2,
        "t_statistic": t_stat,
        "raw_p_value": raw_p
    })

    return pd.DataFrame(rows)

In [25]:
pairwise_results = pairwise_ttests(
    experiment_m,
    value_col="viewing_time",
    group_col="group"
)

pairwise_results.head()

,group_1,group_2,t_statistic,raw_p_value
0,Test1,Test2,-2.845693,0.004686
1,Test1,Test3,-1.917291,0.055942
2,Test1,Test4,-2.799258,0.005391
3,Test1,Test5,0.279632,0.779907
4,Test1,Test6,-3.281141,0.001125
